In [1]:
!pip install --upgrade torchao peft transformers -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 69.2 MB/s eta 0:00:00


In [2]:
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
    stream=sys.stdout
)
log = logging.getLogger(__name__)

In [3]:
import logging
from typing import List, Dict
from datasets import load_dataset

log = logging.getLogger(__name__)

class MathDatasetBuilder:
    def __init__(self, dataset_name: str, max_prompt_len: int = 512):
        self.dataset_name = dataset_name
        self.max_prompt_len = max_prompt_len
        self.system_prompt = (
            "You are a mathematics expert. "
            "Solve the problem step by step and enclose your final answer in \\boxed{}."
        )

    def format_chat_prompt(self, problem: str, tokenizer) -> str:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": problem},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def _process_dataset(self, dataset_split, tokenizer) -> List[Dict]:
        """Hàm xử lý chung để encode prompt và trích xuất gold_answer."""
        out = []
        for item in dataset_split:
            problem = item.get("problem", "")
            gold_answer = item.get("answer", "")

            if not problem: continue

            txt = self.format_chat_prompt(problem, tokenizer)
            ids = tokenizer.encode(txt, add_special_tokens=False)

            if len(ids) <= self.max_prompt_len:
                out.append({
                    "prompt": txt,
                    "prompt_ids": ids,
                    "gold_answer": gold_answer
                })

        return out

    def load_train_data(self, tokenizer) -> List[Dict]:
        log.info(f"Đang load data train từ {self.dataset_name}...")

        ds = load_dataset(self.dataset_name, split="test")

        # 100 sample đầu
        train_ds = ds.select(range(100))

        out = self._process_dataset(train_ds, tokenizer)
        log.info(f"Đã load {len(out)} training examples.")
        return out

    def load_val_data(self, tokenizer) -> List[Dict]:
        log.info(f"Đang load data validation từ {self.dataset_name}...")
        ds = load_dataset(self.dataset_name, split="test")

        # 400 sample
        val_ds = ds.select(range(100, len(ds)))

        out = self._process_dataset(val_ds, tokenizer)
        log.info(f"Đã load {len(out)} validation examples.")
        return out

if __name__ == "__main__":
    # test
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Math-1.5B-Instruct")

    builder = MathDatasetBuilder("HuggingFaceH4/MATH-500")

    train_data = builder.load_train_data(tokenizer)
    val_data = builder.load_val_data(tokenizer)

    print("Example:")
    print("Prompt:", train_data[0]["prompt"][:100], "...")
    print("Gold Answer:", train_data[0]["gold_answer"])

13:50:25 | INFO | NumExpr defaulting to 2 threads.
13:50:26 | INFO | TensorFlow version 2.20.0 available.
13:50:26 | INFO | JAX version 0.7.2 available.
13:50:53 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
13:50:53 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


13:50:53 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

13:50:53 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


...

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

13:50:57 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/MATH-500.py "HTTP/1.1 404 Not Found"
13:50:58 | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/HuggingFaceH4/MATH-500/HuggingFaceH4/MATH-500.py "HTTP/1.1 404 Not Found"
13:50:58 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/HuggingFaceH4/MATH-500/revision/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be "HTTP/1.1 200 OK"
13:50:58 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/.huggingface.yaml "HTTP/1.1 404 Not Found"
13:50:58 | INFO | HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=HuggingFaceH4/MATH-500 "HTTP/1.1 200 OK"
13:50:58 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/HuggingFaceH4/MATH-500/tree/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/data?recursive=true&expand=

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

13:50:59 | INFO | Đã load 100 training examples.
13:50:59 | INFO | Đang load data validation từ HuggingFaceH4/MATH-500...
13:50:59 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
13:50:59 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/HuggingFaceH4/MATH-500/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/README.md "HTTP/1.1 200 OK"
13:51:00 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/MATH-500.py "HTTP/1.1 404 Not Found"
13:51:00 | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/HuggingFaceH4/MATH-500/HuggingFaceH4/MATH-500.py "HTTP/1.1 404 Not Found"
13:51:00 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be/.huggingface.yaml "HTTP/1.1 404 Not Found"
13:51:00 |

In [4]:
import logging
import torch
from transformers import AutoModelForCausalLM

log = logging.getLogger(__name__)

def load_policy_and_ref_models(model_name: str, dtype: torch.dtype, device_map: str = "auto"):
    """Loads the trainable Actor (Policy) and frozen Reference models."""

    log.info(f"Loading Actor model (Policy): {model_name}")
    policy_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map=device_map,
        trust_remote_code=True
    )
    policy_model.gradient_checkpointing_enable()
    policy_model.train()

    log.info(f"Loading Reference model (Frozen): {model_name}")
    ref_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map=device_map,
        trust_remote_code=True
    )
    ref_model.eval()

    # Đóng băng toàn bộ trọng số của Reference Model
    for param in ref_model.parameters():
        param.requires_grad_(False)

    return policy_model, ref_model

In [5]:
import re
import torch
import logging
from transformers import AutoTokenizer, AutoModel

log = logging.getLogger(__name__)

class RuleBasedRewardScorer:
    def __init__(self, device: torch.device):
        self.device = device
        log.info("Đang khởi tạo Rule-Based Reward Scorer...")

    def extract_boxed_answer(self, text: str) -> str:

        # tìm pattern \boxed{}
        matches = re.findall(r'\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}', text)
        if matches:
            return matches[-1].strip()

        # fallback
        fallback = re.findall(r'The answer is:?\s*\$?([^\$\n]+)\$?', text, re.IGNORECASE)
        if fallback:
            return fallback[-1].strip()

        return ""

    def normalize_answer(self, ans: str) -> str:
        if not ans: return ""
        ans = ans.replace(" ", "").lower().replace(",", "")
        ans = ans.rstrip('.')
        return ans

    def get_scores(self, responses: list[str], gold_answer: str) -> torch.Tensor:
        """
        - Đúng đáp án: +1.0 (Phần thưởng tối đa)
        - Có \boxed{} nhưng tính sai: -0.5 (Phạt nhẹ)
        - Không đưa ra được kết luận / Lạc đề: -1.0 (Phạt nặng)
        """
        rewards = []
        norm_gold = self.normalize_answer(gold_answer)

        for resp in responses:
            pred_ans = self.extract_boxed_answer(resp)

            if not pred_ans:
                rewards.append(-1.0)
                continue

            norm_pred = self.normalize_answer(pred_ans)

            if norm_pred == norm_gold:
                rewards.append(1.0)
            else:
                rewards.append(-0.5)

        return torch.tensor(rewards, dtype=torch.float32, device=self.device)


class RewardModelScorer:
    def __init__(self, rm_name: str, dtype: torch.dtype, device: str):
        self.device = device
        log.info(f"Đang khởi tạo Reward Model {rm_name}...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            rm_name,
            trust_remote_code=True
        )
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        self.model = AutoModel.from_pretrained(
            rm_name,
            torch_dtype=dtype,
            trust_remote_code=True,
            device_map=device
        )
        self.model.eval()

    def get_reward(self, questions: list[str], responses: list[str]) -> torch.Tensor:

        rewards = []
        for q, r in zip(questions, responses):
            messages = [
                {"role": "user", "content": q},
                {"role": "assistant", "content": r}
            ]

            # Format
            text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )

            inputs = self.tokenizer(text, return_tensors="pt").to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs)

                if hasattr(outputs, "score"):
                    score = outputs.score[0].item()
                elif hasattr(outputs, "logits"):
                    score = outputs.logits[0, 0].item()
                else:
                    # Fallback
                    score = outputs[0][0].item()

            rewards.append(score)

        return torch.tensor(rewards, dtype=torch.float32, device=self.device)

In [6]:
import numpy as np
from typing import List

def compute_lpro_advantages(
    rewards: List[float],
    lengths: List[int],
    lambda_len: float = 0.10,
    eps: float = 1e-8
) -> np.ndarray:

    r = np.array(rewards, dtype=np.float64)
    L = np.array(lengths, dtype=np.float64)

    # Z-score chuẩn hóa
    z_r = (r - r.mean()) / (r.std() + eps)
    z_L = (L - L.mean()) / (L.std() + eps)

    advantages = z_r - lambda_len * z_L
    return advantages

In [7]:
import torch
from typing import Tuple

def compute_dapo_token_loss(
    new_logprobs: torch.Tensor,
    old_logprobs: torch.Tensor,
    advantage: float,
    mask: torch.Tensor,
    eps_low: float = 0.20,
    eps_high: float = 0.28
) -> Tuple[torch.Tensor, int]:

    ratio = torch.exp(new_logprobs - old_logprobs)
    ratio_clipped = torch.clamp(ratio, 1.0 - eps_low, 1.0 + eps_high)

    adv_tensor = torch.full_like(new_logprobs, advantage)
    surrogate1 = ratio * adv_tensor
    surrogate2 = ratio_clipped * adv_tensor

    # Pessimistic bound (min) và đổi dấu vì PyTorch dùng Gradient Descent (minimize)
    token_losses = -torch.min(surrogate1, surrogate2)

    # Chỉ tính loss trên các token không phải padding
    masked_loss = token_losses * mask

    return masked_loss.sum(), mask.sum().item()

In [8]:
import os
import math
import random
import logging
import torch
import numpy as np
from torch.optim import AdamW
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm

# from nexus.models.policy import load_policy_and_ref_models
# from nexus.models.reward import RewardModelScorer, RuleBasedRewardScorer
# from nexus.rl.advantages import compute_lpro_advantages
# from nexus.rl.loss import compute_dapo_token_loss
# from nexus.data.builder import MathDatasetBuilder

log = logging.getLogger(__name__)

class NexusTrainer:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.bfloat16 if cfg.bf16 and torch.cuda.is_bf16_supported() else torch.float32

        self._set_seed()
        self._setup_components()

    def _set_seed(self):
        random.seed(self.cfg.seed)
        np.random.seed(self.cfg.seed)
        torch.manual_seed(self.cfg.seed)

    def _setup_components(self):
        # Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.cfg.model_name, trust_remote_code=True)
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Models
        self.model, self.ref_model = load_policy_and_ref_models(self.cfg.model_name, self.dtype)
        if getattr(self.cfg, "use_lora", False):
            log.info("Finetune LoRA: Đang cấu hình PEFT/LoRA adapter...")

            lora_config = LoraConfig(
                task_type=TaskType.CAUSAL_LM,
                inference_mode=False,
                r=self.cfg.lora_r,
                lora_alpha=self.cfg.lora_alpha,
                lora_dropout=self.cfg.lora_dropout,
                target_modules=self.cfg.lora_target_modules
            )

            self.model = get_peft_model(self.model, lora_config)
            self.model.print_trainable_parameters()

            if hasattr(self.model, "enable_input_require_grads"):
                self.model.enable_input_require_grads()
        else:
            log.info("Full Finetune: Đang cấu hình để fine-tune toàn bộ model...")

        # Setup Reward Scorer
        if getattr(self.cfg, "use_rule_based_rm", True):
            self.rm_scorer = RuleBasedRewardScorer(self.device)
        else:
            self.rm_scorer = RewardModelScorer(self.cfg.rm_model_name, self.dtype, self.device)

    @staticmethod
    def get_resp_log_probs(model, full_ids: torch.Tensor, prompt_len: int, no_grad: bool = False) -> torch.Tensor:
        ctx = torch.no_grad() if no_grad else torch.enable_grad()
        with ctx:
            out = model(input_ids=full_ids)
            lp = torch.log_softmax(out.logits[0], dim=-1)
            resp_ids = full_ids[0, prompt_len:]
            resp_lp = lp[prompt_len - 1 : full_ids.shape[1] - 1]
            return resp_lp.gather(1, resp_ids.unsqueeze(-1)).squeeze(-1)

    # eval per epoch
    def evaluate(self, val_dataset):
        self.model.eval()
        correct = 0
        total = len(val_dataset)

        log.info(f"Đang evaluation trên {total} samples...")

        scorer = self.rm_scorer if isinstance(self.rm_scorer, RuleBasedRewardScorer) else RuleBasedRewardScorer(self.device)

        pbar = tqdm(val_dataset, desc="Evaluating", leave=False)

        for example in pbar:
            prompt_ids = example["prompt_ids"]
            gold_answer = example.get("gold_answer", "")

            prompt_len = len(prompt_ids)
            prompt_t = torch.tensor([prompt_ids], dtype=torch.long, device=self.device)

            with torch.no_grad():
                with torch.amp.autocast('cuda', enabled=self.cfg.bf16):
                    gen_out = self.model.generate(
                        prompt_t,
                        max_new_tokens=self.cfg.max_new_tokens,
                        temperature=0.0,
                        do_sample=False,
                        pad_token_id=self.tokenizer.eos_token_id,
                    )

            resp_ids = gen_out[0, prompt_len:]
            pad_mask = (resp_ids != self.tokenizer.eos_token_id) & (resp_ids != self.tokenizer.pad_token_id)
            actual_len = max(pad_mask.sum().item(), 1)
            resp_text = self.tokenizer.decode(resp_ids[:actual_len], skip_special_tokens=True)

            # score reward
            pred_ans = scorer.extract_boxed_answer(resp_text)
            if scorer.normalize_answer(pred_ans) == scorer.normalize_answer(gold_answer) and gold_answer:
                correct += 1

            pbar.set_postfix({"Acc": f"{(correct/(pbar.n+1))*100:.2f}%"})

            del prompt_t, gen_out
            torch.cuda.empty_cache()

        acc = (correct / total) * 100
        log.info(f"Evaluate xong. Accuracy = {acc:.2f}% ({correct}/{total})")

        self.model.train()
        return acc

    # training loop
    def train(self):
        # Prepare Data & Optimizer
        dataset_builder = MathDatasetBuilder(self.cfg.dataset_name, self.cfg.max_prompt_len)

        train_dataset = dataset_builder.load_train_data(self.tokenizer)
        val_dataset = dataset_builder.load_val_data(self.tokenizer)

        optimizer = AdamW(self.model.parameters(), lr=self.cfg.lr, weight_decay=self.cfg.weight_decay)
        total_steps = self.cfg.num_epochs * math.ceil(len(train_dataset) / self.cfg.grad_accum)
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(self.cfg.warmup_ratio * total_steps), total_steps)

        os.makedirs(self.cfg.output_dir, exist_ok=True)
        global_step, acc_loss, acc_reward = 0, 0.0, 0.0

        # Training Loop
        for epoch in range(self.cfg.num_epochs):
            random.shuffle(train_dataset)
            log.info(f"Epoch {epoch + 1}/{self.cfg.num_epochs} (TRAIN)")

            pbar = tqdm(enumerate(train_dataset), total=len(train_dataset), desc=f"Epoch {epoch + 1}/{self.cfg.num_epochs}")

            for idx, example in pbar:
                prompt_ids = example["prompt_ids"]
                prompt_txt = example["prompt"]
                gold_answer = example.get("gold_answer", "")

                prompt_len = len(prompt_ids)
                prompt_t = torch.tensor([prompt_ids], dtype=torch.long, device=self.device)

                # Generate responses
                self.model.eval()
                with torch.no_grad():
                    gen_out = self.model.generate(
                        prompt_t.expand(self.cfg.G, -1),
                        max_new_tokens=self.cfg.max_new_tokens,
                        temperature=self.cfg.temperature,
                        do_sample=True,
                        pad_token_id=self.tokenizer.eos_token_id,
                    )
                self.model.train()

                # process generations
                lengths, masks, full_seqs, resp_texts = [], [], [], []
                for i in range(self.cfg.G):
                    resp_ids = gen_out[i, prompt_len:]
                    pad_mask = (resp_ids != self.tokenizer.eos_token_id) & (resp_ids != self.tokenizer.pad_token_id)
                    actual_len = max(pad_mask.sum().item(), 1)

                    lengths.append(actual_len)
                    masks.append(pad_mask.to(self.device))
                    full_seqs.append(gen_out[i])
                    resp_texts.append(self.tokenizer.decode(resp_ids[:actual_len], skip_special_tokens=True))

                # Reward
                if isinstance(self.rm_scorer, RuleBasedRewardScorer):
                    rewards = self.rm_scorer.get_scores(resp_texts, gold_answer)
                else:
                    rewards = self.rm_scorer.get_reward([prompt_txt]*self.cfg.G, resp_texts)

                if not isinstance(rewards, torch.Tensor):
                    rewards = torch.tensor(rewards, dtype=torch.float32, device=self.device)

                if torch.all(rewards == rewards[0]):
                    del gen_out, prompt_t
                    torch.cuda.empty_cache()
                    continue

                advs = compute_lpro_advantages(rewards.tolist(), lengths, self.cfg.lambda_len)

                # Compute loss
                total_loss_sum = torch.tensor(0.0, device=self.device)
                total_n_tokens = 0

                for i in range(self.cfg.G):
                    seq = full_seqs[i].unsqueeze(0).to(self.device)[:, :prompt_len + lengths[i]]
                    mask_i = masks[i][:lengths[i]]

                    with torch.amp.autocast('cuda', enabled=self.cfg.bf16):
                        new_lp = self.get_resp_log_probs(self.model, seq, prompt_len, no_grad=False)
                    old_lp = self.get_resp_log_probs(self.ref_model, seq, prompt_len, no_grad=True)

                    loss_sum, n_valid = compute_dapo_token_loss(
                        new_lp, old_lp.detach(), float(advs[i]), mask_i, self.cfg.eps_low, self.cfg.eps_high
                    )
                    total_loss_sum += loss_sum
                    total_n_tokens += n_valid

                if total_n_tokens == 0:
                    del gen_out, prompt_t, seq, new_lp, old_lp, loss_sum
                    torch.cuda.empty_cache()
                    continue

                # Backprop
                loss = total_loss_sum / total_n_tokens / self.cfg.grad_accum
                loss.backward()

                acc_loss += loss.item() * self.cfg.grad_accum
                acc_reward += rewards.float().mean().item()

                pbar.set_postfix({
                    "loss": f"{loss.item() * self.cfg.grad_accum:.4f}",
                    "reward": f"{rewards.float().mean().item():.2f}"
                })

                if (idx + 1) % self.cfg.grad_accum == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1

                    if global_step % self.cfg.log_steps == 0:
                        log.info(f"Step {global_step} | Loss: {acc_loss/self.cfg.log_steps:.4f} | RM Score: {acc_reward/(self.cfg.log_steps*self.cfg.grad_accum):.3f}")
                        acc_loss, acc_reward = 0.0, 0.0

                    if global_step % self.cfg.save_steps == 0:
                        self.model.save_pretrained(os.path.join(self.cfg.output_dir, f"ckpt-{global_step}"))
                        self.tokenizer.save_pretrained(os.path.join(self.cfg.output_dir, f"ckpt-{global_step}"))

                del gen_out, prompt_t, seq, new_lp, old_lp, loss_sum, total_loss_sum
                torch.cuda.empty_cache()

            log.info(f"Hoàn thành Train Epoch {epoch + 1}. Bắt đầu Eval...")
            self.evaluate(val_dataset)

        log.info("Training hoàn tất. Đang lưu model...")

        # save
        self.model.save_pretrained(self.cfg.output_dir)
        self.tokenizer.save_pretrained(self.cfg.output_dir)

        # push to hub
        if getattr(self.cfg, "push_to_hub", False):
            if not self.cfg.hf_token:
                log.error("Chưa cung cấp hf_token trong Config.")
            else:
                log.info(f"Đang đẩy model lên Hugging Face Hub ({self.cfg.hub_repo_id})...")
                try:
                    # weights
                    self.model.push_to_hub(
                        self.cfg.hub_repo_id,
                        token=self.cfg.hf_token,
                        commit_message="Upload Nexus Qwen-Math weights"
                    )
                    # tokenizer
                    self.tokenizer.push_to_hub(
                        self.cfg.hub_repo_id,
                        token=self.cfg.hf_token,
                        commit_message="Upload tokenizer"
                    )
                    log.info("Đã đẩy model lên Hugging Face Hub thành công!")
                except Exception as e:
                    log.error(f"Có lỗi xảy ra khi đẩy lên Hub: {e}")

13:51:02 | WARNING | Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [ ]:
import json
import logging
import os

# from nexus.nexus_trainer.trainer import NexusTrainer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger(__name__)

class Config:
    def __init__(self):
        self.model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
        self.rm_model_name = "Qwen/Qwen2.5-Math-RM-72B"
        self.output_dir = "./nexus-1.5b"
        self.hub_repo_id = "YOUR_HF_USERNAME/nexus-1.5b"

        self.dataset_name = "HuggingFaceH4/MATH-500"
        self.max_prompt_len = 512

        self.G = 4
        self.max_new_tokens = 1024
        self.temperature = 0.7
        self.top_p = 0.95

        self.eps_low = 0.20
        self.eps_high = 0.28
        self.lambda_len = 0.05
        self.eps_r = 1e-8
        self.eps_l = 1e-8

        self.use_lora = True
        self.use_rule_based_rm = True

        self.lora_r = 16
        self.lora_alpha = 32
        self.lora_dropout = 0.05
        self.lora_target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

        self.num_epochs = 5

        self.lr = 2e-4
        self.weight_decay = 1e-2
        self.warmup_ratio = 0.05
        self.grad_clip = 1.0
        self.grad_accum = 8
        self.bf16 = True

        self.log_steps = 10
        self.save_steps = 100
        self.seed = 42

        self.push_to_hub = True
        self.hf_token = ""


def main():
    cfg = Config()

    cfg_dict = {k: v for k, v in cfg.__dict__.items() if not k.startswith('__')}
    log.info("Starting training with config:\n" + json.dumps(cfg_dict, indent=2, default=str))

    trainer = NexusTrainer(cfg)
    trainer.train()

if __name__ == "__main__":
    main()

13:51:04 | INFO | Starting training with config:
{
  "model_name": "Qwen/Qwen2.5-Math-1.5B-Instruct",
  "rm_model_name": "Qwen/Qwen2.5-Math-RM-72B",
  "output_dir": "./nexus-1.5b",
  "hub_repo_id": "YOUR_HF_USERNAME/nexus-1.5b",
  "dataset_name": "HuggingFaceH4/MATH-500",
  "max_prompt_len": 512,
  "G": 4,
  "max_new_tokens": 1024,
  "temperature": 0.7,
  "top_p": 0.95,
  "eps_low": 0.2,
  "eps_high": 0.28,
  "lambda_len": 0.05,
  "eps_r": 1e-08,
  "eps_l": 1e-08,
  "use_lora": true,
  "use_rule_based_rm": true,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "lora_target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "num_epochs": 5,
  "lr": 0.0002,
  "weight_decay": 0.01,
  "warmup_ratio": 0.05,
  "grad_clip": 1.0,
  "grad_accum": 8,
  "bf16": true,
  "log_steps": 10,
  "save_steps": 100,
  "seed": 42,
  "push_to_hub": true,
  "hf_token": ""
}
13:51:05 | INFO | HTTP Request: HEAD https://hugging

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


13:51:07 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
13:51:07 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"
13:51:07 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/model.safetensors "HTTP/1.1 302 Found"
13:51:07 | INFO | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Math-1.5B-Instruct/xet-read-token/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

13:51:36 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
13:51:36 | INFO | Loading Reference model (Frozen): Qwen/Qwen2.5-Math-1.5B-Instruct
13:51:36 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
13:51:36 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"
13:51:36 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
13:51:36 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

13:51:38 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
13:51:38 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/generation_config.json "HTTP/1.1 200 OK"
13:51:38 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
13:51:38 | INFO | Finetune LoRA: Đang cấu hình PEFT/LoRA adapter...
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
13:51:39 | INFO | Đang khởi tạo Rule-Based Reward Scorer...
13:51:39 | INFO | Đang load data train từ HuggingFaceH4/MATH-500...
13:51:39 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceH4/MATH-500/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
13:51:39 | INFO | HTTP Request: HEAD https://huggingface

Epoch 1/5:   1%|          | 1/100 [00:45<1:15:41, 45.87s/it]